# Capítulo 10: Dispositivos de Estado Sólido Avanzados, Superconductividad y Memristores
### Física Electrónica - UTN FRA
*Simulación de transistores JFET (Shockley), Efecto Meissner en superconductores y Lazo de histéresis del Memristor.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)


## 1. Transistor de Efecto de Campo de Unión (JFET)
En la región de saturación ($V_{DS} \ge V_{GS} - V_P$), la corriente de drenador obedece la **Ecuación de Shockley**:
$$I_D = I_{DSS} \left(1 - \frac{V_{GS}}{V_P}\right)^2$$
En la región óhmica ($V_{DS} < V_{GS} - V_P$):
$$I_D = \frac{2 I_{DSS}}{V_P^2} \left[ (V_{GS} - V_P)V_{DS} - \frac{V_{DS}^2}{2} \right]$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

IDSS = 10e-3 # 10 mA
VP = -4.0    # Tensión de estrangulamiento (Volts)

# 1. Curva de Transferencia (ID vs VGS)
vgs_vals = np.linspace(VP, 0, 200)
id_transf = IDSS * (1 - vgs_vals / VP)**2

# 2. Familia de Curvas de Salida (ID vs VDS)
vds = np.linspace(0, 10, 300)
vgs_curvas = [0.0, -1.0, -2.0, -3.0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(vgs_vals, id_transf * 1e3, 'b-', lw=2.5)
ax1.set_title('Curva de Transferencia del JFET (Shockley)')
ax1.set_xlabel('$V_{GS}$ [V]')
ax1.set_ylabel('$I_D$ [mA]')
ax1.grid(True, linestyle='--')

for vgs in vgs_curvas:
    vds_sat = vgs - VP
    id_curve = []
    for v in vds:
        if v < vds_sat:
            i = (2 * IDSS / (VP**2)) * ((vgs - VP)*v - 0.5 * (v**2))
        else:
            i = IDSS * (1 - vgs / VP)**2
        id_curve.append(max(0, i))
    ax2.plot(vds, np.array(id_curve) * 1e3, lw=2, label=f'$V_{{GS}} = {vgs:.1f}$ V')

# Parábola de estrangulamiento (límite óhmica / saturación)
vds_pinch = np.linspace(0, -VP, 100)
id_pinch = IDSS * (vds_pinch / (-VP))**2
ax2.plot(vds_pinch, id_pinch * 1e3, 'k--', label='Límite de Estrangulamiento ($V_{DS,sat}$)')
ax2.set_title('Curvas de Salida del JFET ($I_D$ vs $V_{DS}$)')
ax2.set_xlabel('$V_{DS}$ [V]')
ax2.set_ylabel('$I_D$ [mA]')
ax2.grid(True, linestyle='--')
ax2.legend()

plt.tight_layout()
plt.show()

## 2. Superconductividad: Penetración de London y Efecto Meissner
En el interior de un superconductor, el campo magnético decae exponencialmente según la ecuación de London:
$$B(x) = B_0 \exp(-x / \lambda_L)$$
donde $\lambda_L$ es la longitud de penetración de London (típicamente $50\text{ nm}$ a $100\text{ nm}$).

In [ ]:
lambda_L = 50e-9 # 50 nm
x = np.linspace(0, 300e-9, 500) # De 0 a 300 nm hacia el interior
B0 = 1.0 # 1 Tesla superficial
B_x = B0 * np.exp(-x / lambda_L)

plt.figure(figsize=(7, 4))
plt.plot(x * 1e9, B_x, 'r-', lw=2.5, label='Campo Magnético $B(x)$')
plt.axvline(lambda_L * 1e9, color='k', ls=':', label=f'$\lambda_L = {lambda_L*1e9:.0f}$ nm')
plt.title('Efecto Meissner: Decaimiento Exponencial del Campo de London')
plt.xlabel('Profundidad en el Superconductor [nm]')
plt.ylabel('Campo Magnético Normalizado $B/B_0$')
plt.grid(True, linestyle='--')
plt.legend()
plt.show()

## 3. El Memristor de Leon Chua: Lazo de Histéresis Estrangulado
El memristor exhibe una resistencia dependiente de la carga que ha circulado:
$$M(q) = R_{\text{on}}\frac{w(t)}{D} + R_{\text{off}}\left(1 - \frac{w(t)}{D}\right)$$
Bajo una señal sinusoidal de tensión, la curva $I-V$ forma la firma característica de **ocho estrangulado** (*pinched hysteresis loop*) en el origen.

In [ ]:
Ron, Roff = 100.0, 10000.0
D = 10e-9 # 10 nm de oxido de titanio
mu_v = 1e-14 # movilidad ionica

t = np.linspace(0, 2*np.pi, 2000)
freq = 1.0
v = np.sin(freq * t)

# Integracion de la posicion de frontera w(t)
w = 0.5 * D # posicion inicial media
i_vals = []
dt = t[1] - t[0]
for val_v in v:
    M = Ron * (w/D) + Roff * (1 - w/D)
    i = val_v / M
    i_vals.append(i)
    dw = (mu_v * Ron / D) * i * dt
    w = np.clip(w + dw, 0, D)

plt.figure(figsize=(7, 5))
plt.plot(v, np.array(i_vals)*1e3, 'm-', lw=2.5)
plt.title('Lazo de Histéresis Estrangulado del Memristor (Curva $I-V$)')
plt.xlabel('Tensión Aplicada $V$ [V]')
plt.ylabel('Corriente $I$ [mA]')
plt.grid(True, linestyle='--')
plt.show()